# APD benchmark — 2. Coordinator scoring (after all three hand in)

Each operator shares their `apd_outputs_<model>` folder with you (Viewer). Add each one to My Drive with **Add shortcut**. If you deleted `apd_private` from Drive, upload `test_labels.csv` in cell 2 (a 1 KB label file, not data).

In [ ]:
#@title 1. Mount Drive and load the code
# Mount Google Drive (NOT upload) and get the project code.
from google.colab import drive
drive.mount("/content/drive")

import os, sys, glob, shutil, subprocess
REPO = "/content/research"
if not os.path.exists(REPO):
    token = None
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")      # optional Colab secret (key icon on the left)
    except Exception:
        pass
    if token:
        subprocess.run(["git", "clone", "-q", f"https://{token}@github.com/whiteraay/research.git", REPO], check=True)
        subprocess.run(["git", "-C", REPO, "remote", "set-url", "origin", "https://github.com/whiteraay/research.git"])
        print("code: cloned from GitHub")
    else:
        cands = ["/content/drive/MyDrive/apd_benchmark/code/research",
                 *glob.glob("/content/drive/.shortcut-targets-by-id/*/apd_benchmark/code/research")]
        src = next((c for c in cands if os.path.exists(c)), None)
        if src is None:
            raise SystemExit("No GITHUB_TOKEN secret and no code copy at MyDrive/apd_benchmark/code/research. "
                             "Add the apd_benchmark shortcut to My Drive (operators) or add a GITHUB_TOKEN secret.")
        shutil.copytree(src, REPO)
        print("code: copied from", src)
sys.path.insert(0, REPO + "/src")
from apd_bench import colab as apd

In [ ]:
#@title 2. Locate the private test labels
from pathlib import Path
LABELS = apd.PRIVATE / "test_labels.csv"
if not LABELS.exists():
    from google.colab import files
    up = files.upload()              # choose test_labels.csv
    LABELS = Path("/content") / next(iter(up))
print("labels:", LABELS)

In [ ]:
#@title 3. Collect the three outputs
apd.coordinator_collect()

In [ ]:
#@title 4. Contract check per model
for m in apd.MODELS:
    d = apd.REPO / "outputs" / m
    if d.exists():
        print("==", m); apd.check_contract(d)

In [ ]:
#@title 5. Score: AUROC, strata, pairwise differences, confound floor
apd.coordinator_score(LABELS)

In [ ]:
#@title 6. Show tables
import pandas as pd
r = apd.REPO / "results/arm_b"
for f in ["summary.csv", "strata.csv", "pairwise.csv"]:
    print("\n==", f); print(pd.read_csv(r / f).round(3).to_string(index=False))

Next: rename `outputs/claude|gpt|gemini` to A/B/C for blind rating (rubric in the Agent Prompt Package doc), and have each operator reproduce another operator's `run.sh --reuse-cache` for the reproducibility score.